In [1]:
# Cài đặt PySpark
%pip install pyspark

In [2]:
# Import các thư viện cần thiết
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import os

# Khởi tạo Spark Context
conf = SparkConf().setAppName("MovieRatingsAnalysis").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)
spark = SparkSession.builder.appName("MovieRatingsAnalysis").getOrCreate()

print("Spark Context đã được khởi tạo thành công!")

Spark Context đã được khởi tạo thành công!


In [3]:
# Đọc dữ liệu từ các file
import os

# Check if running in Google Colab
if 'COLAB_GPU' in os.environ or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    data_path = "/content/"
else:
    data_path = "data/" # For local environment

# Đọc file movies.txt
movies_rdd = sc.textFile(data_path + "movies.txt")
print(f"Số lượng phim: {movies_rdd.count()}")

# Đọc file ratings_1.txt và ratings_2.txt
ratings_1_rdd = sc.textFile(data_path + "ratings_1.txt")
ratings_2_rdd = sc.textFile(data_path + "ratings_2.txt")

# Đọc file users.txt
users_rdd = sc.textFile(data_path + "users.txt")

print(f"Số lượng rating từ file 1: {ratings_1_rdd.count()}")
print(f"Số lượng rating từ file 2: {ratings_2_rdd.count()}")

# Hiển thị một số dòng dữ liệu mẫu
print("\nDữ liệu movies.txt (5 dòng đầu):")
for line in movies_rdd.take(5):
    print(line)

print("\nDữ liệu ratings_1.txt (5 dòng đầu):")
for line in ratings_1_rdd.take(5):
    print(line)

print("\nDữ liệu users.txt (5 dòng đầu):")
for line in users_rdd.take(5):
    print(line)

Số lượng phim: 50
Số lượng rating từ file 1: 84
Số lượng rating từ file 2: 100

Dữ liệu movies.txt (5 dòng đầu):
1001,The Godfather (1972),Crime|Drama
1002,The Shawshank Redemption (1994),Drama
1003,Schindler's List (1993),Biography|Drama|History
1004,Raging Bull (1980),Biography|Drama|Sport
1005,Casablanca (1942),Drama|Romance|War

Dữ liệu ratings_1.txt (5 dòng đầu):
7,1020,4.5,1577836800
23,1015,3.5,1577923200
45,1030,4.0,1578009600
12,1047,3.0,1578096000
38,1012,4.5,1578182400

Dữ liệu users.txt (5 dòng đầu):
1,M,28,3,12345
2,F,35,7,23456
3,M,42,2,34567
4,F,19,10,45678
5,M,31,1,56789


In [4]:
# Parse users.txt: UserID, Gender, Age, Occupation, Zip-code
def parse_user(line):
    parts = line.split(',')
    user_id = int(parts[0])
    gender = parts[1]  # M hoặc F
    age = int(parts[2])
    occupation = int(parts[3])
    zipcode = parts[4]
    return (user_id, gender)

users_parsed = users_rdd.map(parse_user)
print("Users parsed (5 records):")
for user in users_parsed.take(5):
    print(f"UserID: {user[0]}, Gender: {user[1]}")

# Tạo dictionary để tra cứu giới tính theo UserID
users_dict = users_parsed.collectAsMap()
print(f"\nTổng số user trong dictionary: {len(users_dict)}")

Users parsed (5 records):
UserID: 1, Gender: M
UserID: 2, Gender: F
UserID: 3, Gender: M
UserID: 4, Gender: F
UserID: 5, Gender: M

Tổng số user trong dictionary: 50


In [5]:
# Xử lý dữ liệu movies
# Parse movies.txt: MovieID, Title, Genres
def parse_movie(line):
    parts = line.split(',', 2)  # Tách thành 3 phần: ID, Title, Genres
    movie_id = int(parts[0])
    title = parts[1]
    genres = parts[2] if len(parts) > 2 else ""
    return (movie_id, title)

movies_parsed = movies_rdd.map(parse_movie)
print("Movies parsed (5 records):")
for movie in movies_parsed.take(5):
    print(f"MovieID: {movie[0]}, Title: {movie[1]}")

# Tạo dictionary để tra cứu tên phim theo ID
movies_dict = movies_parsed.collectAsMap()
print(f"\nTổng số phim trong dictionary: {len(movies_dict)}")

Movies parsed (5 records):
MovieID: 1001, Title: The Godfather (1972)
MovieID: 1002, Title: The Shawshank Redemption (1994)
MovieID: 1003, Title: Schindler's List (1993)
MovieID: 1004, Title: Raging Bull (1980)
MovieID: 1005, Title: Casablanca (1942)

Tổng số phim trong dictionary: 50


In [6]:
# Xử lý dữ liệu ratings
# Parse ratings: UserID, MovieID, Rating, Timestamp
def parse_rating_with_user(line):
    parts = line.split(',')
    user_id = int(parts[0])
    movie_id = int(parts[1])
    rating = float(parts[2])
    timestamp = int(parts[3])
    return (user_id, movie_id, rating)

# Parse cả 2 file ratings
ratings_1_parsed = ratings_1_rdd.map(parse_rating_with_user)
ratings_2_parsed = ratings_2_rdd.map(parse_rating_with_user)

print("Ratings 1 parsed (5 records):")
for rating in ratings_1_parsed.take(5):
    print(f"UserID: {rating[0]}, MovieID: {rating[1]}, Rating: {rating[2]}")

print("\nRatings 2 parsed (5 records):")
for rating in ratings_2_parsed.take(5):
    print(f"UserID: {rating[0]}, MovieID: {rating[1]}, Rating: {rating[2]}")

# Gộp 2 RDD ratings lại
all_ratings = ratings_1_parsed.union(ratings_2_parsed)
print(f"\nTổng số ratings từ cả 2 file: {all_ratings.count()}")

Ratings 1 parsed (5 records):
UserID: 7, MovieID: 1020, Rating: 4.5
UserID: 23, MovieID: 1015, Rating: 3.5
UserID: 45, MovieID: 1030, Rating: 4.0
UserID: 12, MovieID: 1047, Rating: 3.0
UserID: 38, MovieID: 1012, Rating: 4.5

Ratings 2 parsed (5 records):
UserID: 12, MovieID: 1012, Rating: 3.5
UserID: 34, MovieID: 1039, Rating: 4.0
UserID: 27, MovieID: 1043, Rating: 4.5
UserID: 8, MovieID: 1020, Rating: 3.0
UserID: 19, MovieID: 1050, Rating: 4.0

Tổng số ratings từ cả 2 file: 184


In [7]:
# Thêm thông tin giới tính vào ratings
# all_ratings: (user_id, movie_id, rating)
# Thêm giới tính: (user_id, movie_id, rating, gender)
def add_gender_to_rating(record):
    user_id, movie_id, rating = record
    gender = users_dict.get(user_id, "Unknown")
    return (movie_id, (rating, gender))  # (movie_id, (rating, gender))

ratings_with_gender = all_ratings.map(add_gender_to_rating)

print("Ratings with gender (10 records):")
for record in ratings_with_gender.take(10):
    print(f"MovieID: {record[0]}, Rating: {record[1][0]}, Gender: {record[1][1]}")

# Lọc chỉ những rating có giới tính xác định (M hoặc F)
valid_ratings = ratings_with_gender.filter(lambda x: x[1][1] in ['M', 'F'])

print(f"\nTổng số ratings có giới tính hợp lệ: {valid_ratings.count()}")

# Tách ratings theo giới tính
male_ratings = valid_ratings.filter(lambda x: x[1][1] == 'M').map(lambda x: (x[0], x[1][0]))  # (movie_id, rating)
female_ratings = valid_ratings.filter(lambda x: x[1][1] == 'F').map(lambda x: (x[0], x[1][0]))  # (movie_id, rating)

print(f"Số ratings từ nam: {male_ratings.count()}")
print(f"Số ratings từ nữ: {female_ratings.count()}")

Ratings with gender (10 records):
MovieID: 1020, Rating: 4.5, Gender: M
MovieID: 1015, Rating: 3.5, Gender: M
MovieID: 1030, Rating: 4.0, Gender: M
MovieID: 1047, Rating: 3.0, Gender: F
MovieID: 1012, Rating: 4.5, Gender: F
MovieID: 1050, Rating: 3.5, Gender: F
MovieID: 1037, Rating: 4.0, Gender: M
MovieID: 1040, Rating: 3.0, Gender: M
MovieID: 1025, Rating: 4.5, Gender: F
MovieID: 1010, Rating: 3.5, Gender: M

Tổng số ratings có giới tính hợp lệ: 184
Số ratings từ nam: 92
Số ratings từ nữ: 92


In [8]:
# Tính điểm trung bình cho từng giới tính
# Tính stats cho nam
male_stats = male_ratings.map(lambda x: (x[0], (x[1], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
male_averages = male_stats.map(lambda x: (x[0], x[1][0] / x[1][1]))  # (movie_id, average_rating)

# Tính stats cho nữ
female_stats = female_ratings.map(lambda x: (x[0], (x[1], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
female_averages = female_stats.map(lambda x: (x[0], x[1][0] / x[1][1]))  # (movie_id, average_rating)

# Join male và female averages để có cả 2 điểm cho mỗi phim hoặc ít nhất 1 điểm
movie_gender_ratings = male_averages.fullOuterJoin(female_averages)  # (movie_id, (Option[male_avg], Option[female_avg]))

# Thêm tên phim vào kết quả và xử lý các giá trị None
def add_movie_title_with_na(record):
    movie_id, (male_avg_option, female_avg_option) = record
    movie_title = movies_dict.get(movie_id, f"Unknown Movie {movie_id}")

    male_avg_display = f"{male_avg_option:.2f}" if male_avg_option is not None else "NA"
    female_avg_display = f"{female_avg_option:.2f}" if female_avg_option is not None else "NA"

    return (movie_id, (movie_title, male_avg_display, female_avg_display))

final_results = movie_gender_ratings.map(add_movie_title_with_na)

# Hiển thị kết quả theo định dạng yêu cầu
all_movies = final_results.collect()

for movie_id, (title, male_avg_display, female_avg_display) in all_movies:
    print(f"{title} - Male_Avg: {male_avg_display}, Female_Avg: {female_avg_display}")

Gladiator (2000) - Male_Avg: 3.59, Female_Avg: 3.64
The Terminator (1984) - Male_Avg: 3.93, Female_Avg: 4.14
Lawrence of Arabia (1962) - Male_Avg: 3.55, Female_Avg: 3.31
Mad Max: Fury Road (2015) - Male_Avg: 4.00, Female_Avg: 3.32
No Country for Old Men (2007) - Male_Avg: 3.92, Female_Avg: 3.83
E.T. the Extra-Terrestrial (1982) - Male_Avg: 3.81, Female_Avg: 3.55
Fight Club (1999) - Male_Avg: 3.50, Female_Avg: 3.50
Psycho (1960) - Male_Avg: NA, Female_Avg: 4.00
The Lord of the Rings: The Fellowship of the Ring (2001) - Male_Avg: 4.00, Female_Avg: 3.80
The Godfather: Part II (1974) - Male_Avg: 4.06, Female_Avg: 3.94
The Silence of the Lambs (1991) - Male_Avg: 3.33, Female_Avg: 3.00
Sunset Boulevard (1950) - Male_Avg: 4.33, Female_Avg: 4.50
The Social Network (2010) - Male_Avg: 4.00, Female_Avg: 3.67
The Lord of the Rings: The Return of the King (2003) - Male_Avg: 3.75, Female_Avg: 3.90


In [9]:
# Dọn dẹp tài nguyên
sc.stop()
spark.stop()
print("Đã dừng Spark Context và Spark Session.")

Đã dừng Spark Context và Spark Session.
